# Module 5: Supervised Fine-Tuning (SFT)

> **Course: Training MiniMind LLM from Scratch**  |  *Google Colab NLP Course*

## 🎯 Learning Objectives
In this module you will:
- Understand **why SFT is needed** after pre-training
- Train MiniMind to follow instructions using SFT data
- Compare outputs of the base pre-trained model with the SFT model

---

## 🧠 Motivation: From Word Completion to Instruction Following

Pre-training teaches the model to *predict the next token*, which means given the prompt  
`"The capital of France is"` the model will likely continue with `"Paris…"`.

But users want to *ask questions* and receive *well-formed answers*.  
**Supervised Fine-Tuning (SFT)** bridges this gap by training on `(instruction, response)` pairs,
teaching the model to **follow a conversation format**:

```
[User]      → 解释什么是机器学习
[Assistant] → 机器学习是人工智能的一个分支……
```

The training objective is identical to pre-training (next-token prediction), but the *data format*  
changes — the loss is **only computed on the assistant's response tokens** (instruction tokens are masked).

### SFT Data Format
Each example is a JSON object with a `conversations` key:
```json
{"conversations": [
    {"role": "user",      "content": "你好，请介绍一下自己"},
    {"role": "assistant", "content": "你好！我是MiniMind，一个轻量级语言模型。"}
]}
```


In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import subprocess, sys, os

def run(cmd): return subprocess.run(cmd, shell=True, check=False)

# Install / upgrade dependencies
run("pip install -q torch transformers==4.46.3 modelscope tqdm matplotlib")

# Clone repo if not already present
if not os.path.exists('/content/minimind-colab'):
    run("git clone https://github.com/your-org/minimind-colab /content/minimind-colab")

sys.path.insert(0, '/content/minimind-colab')

# GPU info
run("nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader 2>/dev/null || echo 'No GPU detected'")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 💾 Optional: Mount Google Drive

Uncomment the cell below if you want to persist checkpoints across Colab sessions.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# # Then update save_dir to a Drive path, e.g.:
# # save_dir = '/content/drive/MyDrive/minimind_checkpoints'


## 📥 Download SFT Dataset

In [ ]:
import os

dataset_dir = '/content/minimind-colab/dataset'
os.makedirs(dataset_dir, exist_ok=True)

sft_file = f'{dataset_dir}/sft_t2t_mini.jsonl'
if not os.path.exists(sft_file):
    print("Downloading sft_t2t_mini.jsonl …")
    !modelscope download --dataset gongjy/minimind_dataset sft_t2t_mini.jsonl --local_dir {dataset_dir}
else:
    print(f"✅ SFT dataset already exists: {sft_file}")

# Count lines
with open(sft_file) as f:
    n = sum(1 for _ in f)
print(f"Dataset size: {n:,} samples")

# Check that the pretrain checkpoint exists
save_dir = '/content/minimind-colab/out'
pretrain_ckp = f'{save_dir}/pretrain_768.pth'
if os.path.exists(pretrain_ckp):
    size_mb = os.path.getsize(pretrain_ckp) / 1e6
    print(f"✅ Pretrain checkpoint found: {pretrain_ckp} ({size_mb:.1f} MB)")
else:
    print("⚠️  Pretrain checkpoint NOT found. Run Module 4 first, or download it:")
    print("    !modelscope download --model gongjy/minimind pretrain_768.pth --local_dir /content/minimind-colab/out")


## ⚙️ SFT Training Configuration

In [ ]:
import argparse, os, torch

args = argparse.Namespace(
    save_dir='/content/minimind-colab/out',
    save_weight='full_sft',
    data_path='/content/minimind-colab/dataset/sft_t2t_mini.jsonl',
    hidden_size=768, num_hidden_layers=8, use_moe=0,
    epochs=1, batch_size=8, learning_rate=1e-5,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    dtype='bfloat16', num_workers=2,
    accumulation_steps=1, grad_clip=1.0,
    log_interval=50, save_interval=500,
    max_seq_len=512, from_weight='pretrain',
)
os.makedirs(args.save_dir, exist_ok=True)

print("SFT Training Configuration:")
print("=" * 40)
for k, v in vars(args).items():
    print(f"  {k:<22} = {v}")


In [ ]:
import sys
sys.path.insert(0, '/content/minimind-colab')

from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
from dataset.lm_dataset import SFTDataset
from trainer.trainer_utils import setup_seed, get_model_params, SkipBatchSampler, get_lr, init_model
from contextlib import nullcontext
from torch import optim
from torch.utils.data import DataLoader

setup_seed(42)
lm_config = MiniMindConfig(hidden_size=args.hidden_size, num_hidden_layers=args.num_hidden_layers)
model, tokenizer = init_model(lm_config, args.from_weight,
                               tokenizer_path='/content/minimind-colab/model',
                               save_dir=args.save_dir, device=args.device)

train_ds = SFTDataset(args.data_path, tokenizer, max_length=args.max_seq_len)
optimizer = optim.AdamW(model.parameters(), lr=args.learning_rate, weight_decay=0.01)
dtype_t = torch.bfloat16 if args.dtype == 'bfloat16' else torch.float16
autocast_ctx = (nullcontext() if 'cpu' in args.device
                else torch.cuda.amp.autocast(dtype=dtype_t))
scaler = torch.cuda.amp.GradScaler(enabled=(args.dtype == 'float16'))

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model parameters: {total_params:.2f}M")
print(f"Dataset: {len(train_ds):,} samples")
print(f"Device: {args.device}")


## 📊 SFT Data Format

The SFT dataset uses a **chat/conversation** format. Each sample contains:
- A `conversations` list with alternating `user` and `assistant` turns
- The tokenizer applies a **chat template** that inserts special role tokens
- During training, **only assistant tokens contribute to the loss** — the model is  
  not penalized for "not predicting" the user's prompt correctly.

This masking is the critical difference between SFT and raw pre-training.


In [ ]:
import json

with open(args.data_path) as f:
    samples = [json.loads(l) for l in f.readlines()[:3]]

for i, s in enumerate(samples):
    print(f"── Sample {i+1} " + "─"*50)
    print(json.dumps(s, ensure_ascii=False, indent=2)[:500])
    print()

# Show what the tokenized version looks like
print("── Tokenized (first sample, decoded) " + "─"*30)
ids, labels = train_ds[0]
decoded = tokenizer.decode(ids, skip_special_tokens=False)
print(decoded[:400])
print(f"\ninput_ids shape: {ids.shape}, labels shape: {labels.shape}")
print(f"Masked positions (label=-100): {(labels == -100).sum().item()} / {len(labels)}")


## 🏋️ Training Loop

The SFT loop is nearly identical to pre-training. The key differences are:
1. **Data** — `SFTDataset` returns `(input_ids, labels)` where prompt tokens have `label=-100`
2. **Loss** — `CrossEntropyLoss` automatically ignores `label=-100` positions
3. **Learning rate** — much lower (`1e-5`) to avoid catastrophic forgetting of pre-trained knowledge

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import clear_output
from tqdm.notebook import tqdm
import gc

loss_history, step_history = [], []

def sft_train(epochs=1):
    model.train()
    global_step = 0
    for epoch in range(epochs):
        indices = torch.randperm(len(train_ds)).tolist()
        batch_sampler = SkipBatchSampler(indices, args.batch_size, 0)
        loader = DataLoader(train_ds, batch_sampler=batch_sampler,
                            num_workers=args.num_workers, pin_memory=True)
        iters = len(loader)
        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")

        for step, (input_ids, labels) in enumerate(pbar, start=1):
            global_step += 1
            input_ids = input_ids.to(args.device)
            labels    = labels.to(args.device)

            lr = get_lr(epoch * iters + step, epochs * iters, args.learning_rate)
            for pg in optimizer.param_groups:
                pg['lr'] = lr

            with autocast_ctx:
                res  = model(input_ids, labels=labels)
                loss = (res.loss + res.aux_loss) / args.accumulation_steps

            scaler.scale(loss).backward()

            if step % args.accumulation_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            cur = loss.item() * args.accumulation_steps
            loss_history.append(cur)
            step_history.append(global_step)
            pbar.set_postfix({'loss': f'{cur:.4f}', 'lr': f'{lr:.2e}'})

            if global_step % args.log_interval == 0:
                clear_output(wait=True)
                fig, ax = plt.subplots(figsize=(12, 4))
                ax.plot(step_history, loss_history, 'b-', alpha=0.4, linewidth=0.8, label='Loss')
                if len(loss_history) > 20:
                    w  = 20
                    sm = [sum(loss_history[max(0,i-w):i+1]) / min(i+1, w)
                          for i in range(len(loss_history))]
                    ax.plot(step_history, sm, 'r-', linewidth=2, label='Smoothed (w=20)')
                ax.set_title(f'SFT Training Loss  (step {global_step})')
                ax.set_xlabel('Step'); ax.set_ylabel('Loss')
                ax.legend(); ax.grid(True, alpha=0.3)
                plt.tight_layout(); plt.show()

        # ── save checkpoint ──────────────────────────────────────
        model.eval()
        ckp = f'{args.save_dir}/{args.save_weight}_{lm_config.hidden_size}.pth'
        torch.save({k: v.half().cpu() for k, v in model.state_dict().items()}, ckp)
        print(f"\n✅ Checkpoint saved: {ckp}")
        model.train()

sft_train(args.epochs)


## ⚖️ Before vs After: Pretrain vs SFT Comparison

Now let's **qualitatively** compare the two models side-by-side.

- **Pretrain model** receives raw text and tries to *continue* it
- **SFT model** receives a chat-formatted prompt and produces a *structured reply*

You should observe the SFT model giving more coherent, on-topic answers.

In [ ]:
from transformers import AutoTokenizer

def generate_response(m, tok, prompt, is_pretrain=False, device='cuda',
                      max_new_tokens=120, temperature=0.8):
    m.eval()
    if is_pretrain:
        inp_text = tok.bos_token + prompt
    else:
        conv = [{"role": "user", "content": prompt}]
        inp_text = tok.apply_chat_template(conv, tokenize=False,
                                           add_generation_prompt=True)
    inp = tok(inp_text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = m.generate(
            inp["input_ids"],
            attention_mask=inp["attention_mask"],
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
        )
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)

# Load the original pre-train model for comparison
pretrain_model = MiniMindForCausalLM(lm_config)
pretrain_ckp   = f'{args.save_dir}/pretrain_{lm_config.hidden_size}.pth'
state = torch.load(pretrain_ckp, map_location=args.device)
pretrain_model.load_state_dict(state, strict=False)
pretrain_model = pretrain_model.half().eval().to(args.device)
print(f"Pretrain model loaded from: {pretrain_ckp}")

test_questions = [
    "解释什么是机器学习",
    "如何提高英语水平？",
    "推荐一些健康食谱",
]

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"❓ Question: {q}")
    pt = generate_response(pretrain_model, tokenizer, q,
                           is_pretrain=True, device=args.device)
    sf = generate_response(model, tokenizer, q, device=args.device)
    print(f"🟡 Pretrain : {pt[:200]}")
    print(f"🟢 SFT      : {sf[:200]}")


In [ ]:
# ── Memory cleanup ───────────────────────────────────────────────────────────
import gc, torch

del pretrain_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory freed. Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")


## 📝 Student Exercise

1. **Change the learning rate** (`learning_rate=5e-5`) and observe how quickly/slowly the loss drops compared to `1e-5`. What risk do you run at higher LRs?

2. **Create 5 custom SFT samples** in the conversation format and add them to `sft_t2t_mini.jsonl`. Re-run training and verify the model "remembers" your custom facts.

3. **Measure loss on instruction-only tokens**: modify `SFTDataset` to also compute loss on user turns. How does this affect alignment?

## 💡 Discussion Questions

- Why does SFT use a **lower learning rate** than pre-training?
- What is **catastrophic forgetting**, and how does it apply here?
- If you only have 100 labelled examples, is SFT still useful? What alternatives exist (e.g., LoRA)?
- Compare SFT with **RLHF**: what does SFT miss that RLHF can correct?
